# 🚀 LAB GUIDE — PRODUCTION-GRADE GRAPHRAG VS FLAT RAG

**Thời lượng:** 120 phút  
**Môi trường:** Google Colab (T4 GPU khuyến nghị) + Neo4j AuraDB  
**Dữ liệu:** HackerNoon Tech Company News Data Dump (bản thu gọn do giảng viên cung cấp)  
**Công cụ:** Học viên được dùng AI Coding Agent, nhưng phải tự thiết kế, kiểm thử và giải thích logic.

## 🎯 Mục tiêu
1. Xây dựng Hybrid GraphRAG end-to-end.
2. Xử lý Coreference Resolution, Entity Resolution và Super-node Mitigation.
3. Bulk insert bằng `UNWIND`, không insert từng row.
4. So sánh Flat RAG và GraphRAG bằng Golden Dataset + LLM-as-a-Judge.
5. Đo quality, latency và token usage.
6. Giải thích kiến trúc và failure modes.

> Notebook là **reference lab guide**: có code khung chạy được nhưng vẫn yêu cầu học viên thay prompt/threshold/retrieval policy và thuyết minh lựa chọn.

## ⏳ Timeline

| Phút | Nội dung |
|---|---|
| 00–15 | Setup, load, dedup, chunk, coreference |
| 15–45 | NER/RE, entity resolution, Neo4j bulk insert |
| 45–75 | Flat RAG, graph traversal, hybrid retrieval |
| 75–105 | Golden Dataset, LLM-as-a-Judge, comparison |
| 105–120 | Failure-mode tests, bonus, export, thuyết minh |

### Scale guard
Trong lab 2 giờ, không nên gửi toàn bộ 350MB qua LLM. Mặc định dùng subset:
- `LAB_MAX_ARTICLES = 1500`
- `LAB_MAX_CHUNKS = 3000`
- `EXTRACTION_MAX_CHUNKS = 400`

Kiến trúc phải scale được; volume trong giờ lab chỉ dùng để chứng minh pipeline.

# PHẦN 1 — SETUP & PREPROCESSING

### Secrets trên Colab
Tạo:
- `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`
- `GROQ_API_KEY`, `GROQ_MODEL`
- `HF_TOKEN` để stream dataset từ Hugging Face
- cho judge: `JUDGE_PROVIDER`, `JUDGE_MODEL`, và `OPENAI_API_KEY` nếu dùng OpenAI

Không hard-code API key vào notebook nộp bài.

In [1]:
#@title 1.1 — Install
%pip -q install neo4j pandas numpy pyarrow sentence-transformers faiss-cpu groq openai tqdm networkx datasets python-dotenv

Note: you may need to restart the kernel to use updated packages.


C:\Users\thanh\OneDrive\Documents\Máy tính\labvin\Day19_1\K4-Track3-Lab19-GraphRAG\.venv\Scripts\python.exe: No module named pip


In [2]:
#@title 1.2 — Imports & config
import os, re, json, time, random, hashlib, unicodedata
from pathlib import Path
from collections import defaultdict, Counter, deque
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
import faiss

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 120)

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if not IN_COLAB:
    from dotenv import load_dotenv
    load_dotenv()

def get_secret(name, default=None):
    if IN_COLAB:
        try:
            from google.colab import userdata
            value = userdata.get(name)
            if value is not None:
                return value
        except Exception:
            pass
    return os.environ.get(name, default)

NEO4J_URI = get_secret("NEO4J_URI", "")
NEO4J_USER = get_secret("NEO4J_USER") or get_secret("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = get_secret("NEO4J_PASSWORD", "")
NEO4J_DATABASE = get_secret("NEO4J_DATABASE", "neo4j")

GROQ_API_KEY = get_secret("GROQ_API_KEY", "")
GROQ_MODEL = get_secret("GROQ_MODEL", "")

JUDGE_PROVIDER = get_secret("JUDGE_PROVIDER", "openai").lower()
JUDGE_MODEL = get_secret("JUDGE_MODEL", "")
OPENAI_API_KEY = get_secret("OPENAI_API_KEY", "")
HF_TOKEN = get_secret("HF_TOKEN", "")

DATA_PATH = "/content/hackernoon_subset.csv" if IN_COLAB else ".cache/hackernoon_subset.csv"
LAB_MAX_ARTICLES = 1500
LAB_MAX_CHUNKS = 3000
EXTRACTION_MAX_CHUNKS = 200  # giam tu 400 (spec) xuong 200 de vua ngan sach TPD free tier cua Groq
CHUNK_WORDS = 220
CHUNK_OVERLAP_WORDS = 40

C:\Users\thanh\OneDrive\Documents\Máy tính\labvin\Day19_1\K4-Track3-Lab19-GraphRAG\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1.2b — (Chạy Local? Bỏ qua) Clone repo GitHub để lưu output đúng chỗ trên Colab

Chỉ cần khi chạy trên **Google Colab**. Thêm Colab Secrets:
- `GH_TOKEN`: GitHub Personal Access Token (repo scope)
- `GH_REPO`: dạng `username/reponame`
- `GH_USER`, `GH_EMAIL`: tuỳ chọn, dùng để commit

Nếu không có `GH_TOKEN`/`GH_REPO`, notebook vẫn chạy bình thường, nhưng bạn phải tự tải CSV kết quả về máy và bỏ vào đúng thư mục `data/` và `outputs/` của repo.

In [3]:
#@title 1.2b — Clone repo (Colab) để lưu output vào đúng chỗ
import subprocess

GH_TOKEN = get_secret("GH_TOKEN", "")
GH_REPO = get_secret("GH_REPO", "")   # vd: thanhdatpham938-hub/K4-Track3-Lab19-GraphRAG
GH_USER = get_secret("GH_USER", "")
GH_EMAIL = get_secret("GH_EMAIL", "")

REPO_DIR = "/content/repo"

if not (GH_TOKEN and GH_REPO):
    print("[SKIP] Thieu GH_TOKEN/GH_REPO trong Colab Secrets -> khong clone repo.")
    print("CSV se duoc ghi vao /content/, phai tu tai ve va copy vao repo local.")
else:
    if not Path(REPO_DIR).exists():
        clone_url = f"https://{GH_TOKEN}@github.com/{GH_REPO}.git"
        subprocess.run(["git", "clone", clone_url, REPO_DIR], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "config", "user.name", GH_USER or "colab-bot"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "config", "user.email", GH_EMAIL or "colab@example.com"], check=True)
    for d in ["data", "outputs", "reports"]:
        (Path(REPO_DIR) / d).mkdir(parents=True, exist_ok=True)
    print("OK repo cloned tai", REPO_DIR)


[SKIP] Thieu GH_TOKEN/GH_REPO trong Colab Secrets -> khong clone repo.
CSV se duoc ghi vao /content/, phai tu tai ve va copy vao repo local.


## 1.3 — Download HackerNoon Dataset bằng Hugging Face Streaming

Cell dưới đây stream trực tiếp dataset **`HackerNoon/tech-company-news-data-dump`** và ghi dần ra CSV, nên không cần tải toàn bộ dataset vào RAM.

### Hai cơ chế giới hạn

- `LIMIT_ROWS`: số dòng tối đa.
- `LIMIT_MB`: dung lượng file tối đa.
- `PRIORITIZE_MB = True`: ưu tiên dừng theo dung lượng MB.
- `PRIORITIZE_MB = False`: thanh tiến trình theo số dòng, nhưng **vẫn giữ hard-stop `LIMIT_ROWS`**.

### Lưu ý

- Đặt `HF_TOKEN` trong **Colab Secrets**, không hard-code token vào notebook.
- Nếu dataset yêu cầu quyền truy cập/gated access, hãy mở trang dataset trên Hugging Face và hoàn tất bước **Agree/Request access** trước.
- Sau khi cell hoàn tất, `DATA_PATH` mặc định đã trỏ tới `/content/hackernoon_subset.csv`, nên cell loader kế tiếp có thể chạy trực tiếp.

In [4]:
#@title 1.3 — Stream HackerNoon dataset -> CSV
import csv
import os
from datasets import load_dataset
from tqdm.auto import tqdm

DATASET_NAME = "HackerNoon/tech-company-news-data-dump"
OUTPUT_CSV = "/content/hackernoon_subset.csv" if IN_COLAB else ".cache/hackernoon_subset.csv"
os.makedirs(os.path.dirname(OUTPUT_CSV) or ".", exist_ok=True)

LIMIT_ROWS = 1_000_000
LIMIT_MB = 25

PRIORITIZE_MB = True

if not HF_TOKEN:
    raise ValueError("Thieu HF_TOKEN. Hay them Hugging Face Access Token.")

if Path(OUTPUT_CSV).exists() and os.path.getsize(OUTPUT_CSV) > 0:
    print(f"[SKIP] {OUTPUT_CSV} da ton tai, bo qua download lai.")
    DATA_PATH = OUTPUT_CSV
else:
    print("Dang ket noi luong du lieu (streaming)...")

    try:
        dataset = load_dataset(DATASET_NAME, split="train", streaming=True, token=HF_TOKEN)
        iterator = iter(dataset)
        first_row = next(iterator)
        headers = list(first_row.keys())

        print(f"Dang ghi du lieu vao: {OUTPUT_CSV}")

        rows_written = 0
        total_progress = LIMIT_MB if PRIORITIZE_MB else LIMIT_ROWS
        unit_progress = "MB" if PRIORITIZE_MB else "row"

        with open(OUTPUT_CSV, mode="w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=headers, extrasaction="ignore")
            writer.writeheader()
            writer.writerow(first_row)
            rows_written += 1

            f.flush()
            file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)

            with tqdm(total=total_progress, desc=f"Dang tai ({unit_progress})", unit=unit_progress) as pbar:
                if PRIORITIZE_MB:
                    pbar.n = min(file_size_mb, LIMIT_MB)
                    pbar.refresh()
                else:
                    pbar.update(1)

                for row in iterator:
                    writer.writerow(row)
                    rows_written += 1

                    should_check_size = (
                        PRIORITIZE_MB
                        and (rows_written % 100 == 0 or file_size_mb >= LIMIT_MB * 0.95)
                    )

                    if should_check_size:
                        f.flush()
                        file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
                        pbar.n = min(round(file_size_mb, 2), LIMIT_MB)
                        pbar.refresh()
                    elif not PRIORITIZE_MB:
                        pbar.update(1)

                    if PRIORITIZE_MB and file_size_mb >= LIMIT_MB:
                        print()
                        print(f"[DUNG] Da dat gioi han dung luong: {file_size_mb:.2f} MB (Tong: {rows_written:,} dong)")
                        break

                    if rows_written >= LIMIT_ROWS:
                        f.flush()
                        file_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
                        print()
                        print(f"[DUNG] Da dat gioi han so dong: {rows_written:,} dong (Dung luong: {file_size_mb:.2f} MB)")
                        break

            f.flush()

        final_size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
        print(f"Hoan thanh: {os.path.abspath(OUTPUT_CSV)}")
        print(f"   Rows: {rows_written:,}")
        print(f"   Size: {final_size_mb:.2f} MB")

        DATA_PATH = OUTPUT_CSV

    except StopIteration:
        raise RuntimeError("Dataset stream rong: khong lay duoc dong dau tien.")
    except Exception as e:
        print()
        print(f"Co loi xay ra: {e}")
        print("Kiem tra: (1) HF_TOKEN, (2) quyen Agree/Access tren Hugging Face, (3) ket noi mang.")
        raise

[SKIP] .cache/hackernoon_subset.csv da ton tai, bo qua download lai.


In [5]:
#@title 1.4 — Neo4j connection + schema
driver = None

def connect_neo4j():
    global driver
    if not NEO4J_URI or not NEO4J_PASSWORD:
        raise ValueError("Thiếu Neo4j secrets.")
    driver = GraphDatabase.driver(
        NEO4J_URI,
        auth=(NEO4J_USER, NEO4J_PASSWORD),
    )
    driver.verify_connectivity()
    print("✅ Neo4j connected.")

def run_cypher(query, **params):
    if driver is None:
        raise RuntimeError("Hãy chạy connect_neo4j() trước.")
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run(query, **params)
        rows = [r.data() for r in result]
        result.consume()
    return rows

def setup_graph_schema():
    for stmt in [
        """
        CREATE CONSTRAINT entity_id IF NOT EXISTS
        FOR (n:Entity) REQUIRE n.id IS UNIQUE
        """,
        """
        CREATE INDEX entity_name_norm IF NOT EXISTS
        FOR (n:Entity) ON (n.name_norm)
        """,
        """
        CREATE INDEX company_name_norm IF NOT EXISTS
        FOR (n:Company) ON (n.name_norm)
        """,
        """
        CREATE INDEX person_name_norm IF NOT EXISTS
        FOR (n:Person) ON (n.name_norm)
        """,
        """
        CREATE INDEX technology_name_norm IF NOT EXISTS
        FOR (n:Technology) ON (n.name_norm)
        """,
    ]:
        run_cypher(stmt)
    print("✅ Schema ready.")

connect_neo4j()
setup_graph_schema()

✅ Neo4j connected.


✅ Schema ready.


In [6]:
#@title 1.5 — Loader + exact dedup + chunking
def norm_space(x):
    return re.sub(r"\s+", " ", str(x or "")).strip()

def sha1(x):
    return hashlib.sha1(str(x).encode("utf-8", errors="ignore")).hexdigest()

def pick_col(df, candidates, required=True):
    lookup = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lookup:
            return lookup[c.lower()]
    if required:
        raise KeyError(f"Missing one of columns: {candidates}")
    return None

def load_news(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in {".jsonl", ".ndjson"}:
        return pd.read_json(path, lines=True)
    if path.suffix.lower() == ".json":
        return pd.read_json(path)
    if path.suffix.lower() in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported: {path.suffix}")

def standardize_news(raw):
    text_col = pick_col(raw, ["text", "content", "article", "body", "story", "description"])
    title_col = pick_col(raw, ["title", "headline"], required=False)
    date_col = pick_col(raw, ["published_date", "date", "published_at", "created_at"], required=False)
    id_col = pick_col(raw, ["id", "article_id", "story_id", "uuid"], required=False)

    df = pd.DataFrame()
    df["text"] = raw[text_col].fillna("").map(norm_space)
    df["title"] = raw[title_col].fillna("").map(norm_space) if title_col else ""

    if date_col:
        df["published_date"] = (
            pd.to_datetime(raw[date_col], errors="coerce", utc=True)
            .dt.strftime("%Y-%m-%d")
            .fillna("")
        )
    else:
        df["published_date"] = ""

    if id_col:
        df["article_id"] = raw[id_col].astype(str)
    else:
        df["article_id"] = [
            sha1(f"{t}\n{x}")[:20] for t, x in zip(df["title"], df["text"])
        ]

    df = df[df["text"].str.len() >= 80].copy()
    df["dedup_key"] = [
        sha1(norm_space(f"{t}\n{x}").lower())
        for t, x in zip(df["title"], df["text"])
    ]
    before = len(df)
    df = df.drop_duplicates("dedup_key").drop(columns="dedup_key").reset_index(drop=True)
    print(f"Exact dedup: {before:,} -> {len(df):,}")

    if LAB_MAX_ARTICLES and len(df) > LAB_MAX_ARTICLES:
        df = df.sample(LAB_MAX_ARTICLES, random_state=SEED).sort_index().reset_index(drop=True)
    return df

def chunk_text(text, size=220, overlap=40):
    words = norm_space(text).split()
    step = max(1, size - overlap)
    out = []
    for start in range(0, len(words), step):
        part = words[start:start+size]
        if not part:
            break
        out.append(" ".join(part))
        if start + size >= len(words):
            break
    return out

def build_chunks(news_df):
    rows = []
    for r in tqdm(news_df.itertuples(index=False), total=len(news_df), desc="Chunking"):
        for i, text in enumerate(chunk_text(r.text, CHUNK_WORDS, CHUNK_OVERLAP_WORDS)):
            rows.append({
                "chunk_id": f"{r.article_id}::c{i:04d}",
                "article_id": r.article_id,
                "title": r.title,
                "published_date": r.published_date,
                "text": text,
            })
            if LAB_MAX_CHUNKS and len(rows) >= LAB_MAX_CHUNKS:
                return pd.DataFrame(rows)
    return pd.DataFrame(rows)

raw_df = load_news(DATA_PATH)
news_df = standardize_news(raw_df)
chunks_df = build_chunks(news_df)
display(chunks_df.head())

Exact dedup: 20,907 -> 18,860


Chunking:   0%|          | 0/1500 [00:00<?, ?it/s]

Chunking: 100%|██████████| 1500/1500 [00:00<00:00, 20701.98it/s]

,chunk_id,article_id,title,published_date,text
0,b9987134e0e47e507473::c0000,b9987134e0e47e507473,The ACLU is Committed to Protecting Your Personal Information,2023-07-05,Why we remain committed to privacy both internally and externally As privacy and technology continue to evolve we re...
1,391d8c6c09f3574ace59::c0000,391d8c6c09f3574ace59,Crexendo Inc.: Crexendo Adds Jenne Inc. as New Technology Services Brokerage Partner,2023-04-19,Crexendo announced today that Jenne Cloud Services Brokerage is a new Technology Services Brokerage for its VIP Busi...
2,1a10a81ab65b6b84a05e::c0000,1a10a81ab65b6b84a05e,Human Services Technology,2023-06-28,The Human Services Technology (HST) curriculum prepares students for entry-level positions in institutions and agenc...
3,3ac1a0d4fdc98952e374::c0000,3ac1a0d4fdc98952e374,Citi launches token service for institutional clients,2023-09-26,Citi has launched a token service using blockchain technology to offer digital asset solutions for its institutional...
4,e2cb0c7b0a65f7211e15::c0000,e2cb0c7b0a65f7211e15,Spot fraud fast with identity theft protection services that offer up to $1 million in insurance,2023-04-01,While identity theft services can alert you if your personal information appears on the dark web or is misused it''s...


### 🎯 AI Coding Agent Challenge A — Near Dedup
Exact hash không bắt được bài repost/near-duplicate.

Hãy dùng AI Agent thiết kế thêm **MinHash/LSH, SimHash hoặc embedding+ANN**.  
**Không chấp nhận** pairwise cosine `O(N²)` trên toàn dataset.

Trong báo cáo nêu:
1. threshold,
2. false positive,
3. cách audit cặp bị merge.

In [7]:
#@title 1.6 — LLM wrapper: model pool + failover khi cạn quota + retry
from groq import Groq
groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None

# Groq free tier gioi han TOKENS-PER-DAY (TPD) RIENG cho tung model.
# Pipeline nay (coref 400 + extract 400 + eval 25 cau) vuot xa TPD cua 1 model,
# nen ta dung mot POOL: het quota model nay thi tu dong chuyen model ke tiep.
MODEL_POOL = [
    GROQ_MODEL or "openai/gpt-oss-120b",
    "openai/gpt-oss-20b",
    "qwen/qwen3.6-27b",
]
MODEL_POOL = list(dict.fromkeys([m for m in MODEL_POOL if m]))  # bo trung, giu thu tu
_exhausted_models = set()   # model da cham tran TPD trong phien nay

def _is_daily_quota_error(err):
    s = str(err)
    return "rate_limit_exceeded" in s and "TPD" in s

def parse_json_object(text):
    text = str(text).strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)
    a, b = text.find("{"), text.rfind("}")
    if a < 0 or b <= a:
        raise ValueError("No JSON object found.")
    return json.loads(text[a:b+1])

# Cac model "reasoning" (gpt-oss, qwen) mac dinh ro <think>...</think> vao content
# neu khong tat reasoning -> lam nhieu output va sai lech diem Judge. Tham so ten
# khac nhau theo family model nen thu lan luot, bo qua neu API tu choi tham so.
_REASONING_EFFORT_BY_PREFIX = [
    ("openai/gpt-oss", "low"),
    ("qwen/", "none"),
]

def _reasoning_kwargs_for(model):
    for prefix, val in _REASONING_EFFORT_BY_PREFIX:
        if model.startswith(prefix):
            return {"reasoning_effort": val}
    return {}

def _strip_think(text):
    """Luoi an toan: neu model van ro <think>...</think>, cat bo truoc khi dung."""
    return re.sub(r"<think>.*?</think>\s*", "", text, flags=re.S).strip()

def _call_one_model(messages, model, json_mode, max_retries=4):
    """Goi 1 model, retry cho loi tam thoi (TPM). Nem ngay neu la loi TPD."""
    last = None
    for attempt in range(max_retries):
        try:
            kwargs = {"model": model, "messages": messages, "temperature": 0.0}
            if json_mode:
                kwargs["response_format"] = {"type": "json_object"}
            kwargs.update(_reasoning_kwargs_for(model))
            try:
                resp = groq_client.chat.completions.create(**kwargs)
            except Exception as e_param:
                if "reasoning_effort" in str(e_param) and "reasoning_effort" in kwargs:
                    kwargs.pop("reasoning_effort")
                    resp = groq_client.chat.completions.create(**kwargs)
                else:
                    raise
            usage = {}
            if getattr(resp, "usage", None):
                usage = {
                    "prompt_tokens": getattr(resp.usage, "prompt_tokens", None),
                    "completion_tokens": getattr(resp.usage, "completion_tokens", None),
                    "total_tokens": getattr(resp.usage, "total_tokens", None),
                }
            return _strip_think(resp.choices[0].message.content), usage
        except Exception as e:
            last = e
            if _is_daily_quota_error(e):
                raise                      # het quota ngay -> khong retry, de pool doi model
            if attempt == max_retries - 1:
                break
            time.sleep(min(20, 2**attempt + random.random()))
    raise RuntimeError(last)

def groq_chat(messages, model=None, json_mode=False, max_retries=4):
    if groq_client is None:
        raise RuntimeError("Thiếu GROQ_API_KEY.")

    # Uu tien 'model' neu duoc chi dinh, nhung VAN failover qua pool khi model do het TPD
    # (truoc day: co model= la BO QUA pool hoan toan -> day chinh la bug gay 429 o eval)
    pool_rest = [m for m in MODEL_POOL if m not in _exhausted_models and m != model]
    candidates = ([model] if model and model not in _exhausted_models else []) + pool_rest
    if not candidates:
        raise RuntimeError(
            f"Tat ca model trong pool da het quota ngay: {sorted(_exhausted_models)}. "
            "Doi quota reset hoac nang tier."
        )

    last = None
    for m in candidates:
        try:
            return _call_one_model(messages, m, json_mode, max_retries)
        except Exception as e:
            last = e
            if _is_daily_quota_error(e):
                _exhausted_models.add(m)
                print(f"[QUOTA] Model '{m}' het TPD -> chuyen model ke tiep.")
                continue
            raise
    raise RuntimeError(last)

def groq_json(system, user, model=None):
    text, usage = groq_chat(
        [{"role": "system", "content": system},
         {"role": "user", "content": user}],
        model=model,
        json_mode=True,
    )
    return parse_json_object(text), usage

# --- SMOKE TEST: fail fast neu API key/model sai ---
_smoke, _ = groq_json("Return strict JSON only.", 'Return {"ok": true}')
assert _smoke.get("ok") is True, f"Groq smoke test that bai: {_smoke}"
print(f"[SMOKE OK] Model pool: {MODEL_POOL}")

# --- Cache tren dia: cac buoc goi LLM rat dat, khong duoc chay lai vo ich ---
CACHE_DIR = Path(".cache")
CACHE_DIR.mkdir(exist_ok=True)

def cached_df(name, build_fn):
    """Tra ve DataFrame tu cache neu co, nguoc lai chay build_fn roi luu lai."""
    p = CACHE_DIR / f"{name}.pkl"
    if p.exists():
        df = pd.read_pickle(p)
        print(f"[CACHE HIT] {name}: {len(df)} dong (xoa {p} de tinh lai).")
        return df
    df = build_fn()
    df.to_pickle(p)
    print(f"[CACHE SAVE] {name}: {len(df)} dong -> {p}")
    return df


[SMOKE OK] Model pool: ['openai/gpt-oss-120b', 'openai/gpt-oss-20b', 'qwen/qwen3.6-27b']


## 1.7 — Coreference Resolution

Yêu cầu:
- chỉ resolve đại từ khi antecedent rõ trong cùng chunk,
- không invent fact,
- giữ nguyên số/ngày/ticker/product,
- ambiguity → giữ nguyên và log `unresolved_mentions`.

**Failure mode quan trọng:** false coreference → false edge.

In [8]:
#@title 1.7 — Coreference resolution theo batch
COREF_SYSTEM = """
You are a conservative coreference-resolution component for a knowledge-graph pipeline.
Resolve pronouns and generic references only when the antecedent is clearly supported in the same chunk.
Never invent facts. Preserve dates, numbers, tickers and product names.
Return strict JSON only.
""".strip()

def resolve_coref_batch(batch_df):
    payload = [{"chunk_id": r.chunk_id, "text": r.text}
               for r in batch_df.itertuples(index=False)]

    prompt = f"""
Resolve coreferences.

Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "resolved_text": "...",
      "unresolved_mentions": ["..."]
    }}
  ]
}}

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()

    obj, usage = groq_json(COREF_SYSTEM, prompt)
    by_id = {x.get("chunk_id"): x for x in obj.get("items", [])}

    rows = []
    for r in batch_df.itertuples(index=False):
        item = by_id.get(r.chunk_id, {})
        rows.append({
            "chunk_id": r.chunk_id,
            "resolved_text": norm_space(item.get("resolved_text") or r.text),
            "unresolved_mentions": item.get("unresolved_mentions", []),
        })
    return pd.DataFrame(rows), usage

COREF_CKPT = CACHE_DIR / "coref_partial.pkl"

def run_coref(chunks_subset, batch_size=5):
    """Resume duoc: checkpoint sau MOI batch. Quota can giua chung khong mat cong da chay."""
    done = {}
    if COREF_CKPT.exists():
        prev = pd.read_pickle(COREF_CKPT)
        done = {r.chunk_id: r for r in prev.itertuples(index=False)}
        print(f"[RESUME] coref: da co {len(done)} chunk trong checkpoint.")

    out = [pd.DataFrame(
        [{"chunk_id": r.chunk_id,
          "resolved_text": r.resolved_text,
          "unresolved_mentions": r.unresolved_mentions} for r in done.values()]
    )] if done else []

    todo = chunks_subset[~chunks_subset.chunk_id.isin(done.keys())]
    quota_stop = False

    for start in tqdm(range(0, len(todo), batch_size), desc="Coref"):
        batch = todo.iloc[start:start+batch_size]
        try:
            df, _ = resolve_coref_batch(batch)
        except Exception as e:
            if "Tat ca model trong pool da het quota" in str(e):
                print(f"[QUOTA STOP] Dung coref tai batch {start}, giu checkpoint.")
                quota_stop = True
                break
            df = pd.DataFrame({
                "chunk_id": batch["chunk_id"].tolist(),
                "resolved_text": batch["text"].tolist(),
                "unresolved_mentions": [["COREF_BATCH_FAILED"] for _ in range(len(batch))],
            })
        out.append(df)
        pd.concat(out, ignore_index=True).to_pickle(COREF_CKPT)

    result = pd.concat(out, ignore_index=True) if out else pd.DataFrame(
        columns=["chunk_id", "resolved_text", "unresolved_mentions"])

    # Chunk chua kip xu ly -> giu nguyen text goc de pipeline chay tiep duoc
    missing = chunks_subset[~chunks_subset.chunk_id.isin(set(result.chunk_id))]
    if len(missing):
        print(f"[WARN] {len(missing)} chunk chua coref (quota) -> dung text goc.")
        result = pd.concat([result, pd.DataFrame({
            "chunk_id": missing.chunk_id.tolist(),
            "resolved_text": missing.text.tolist(),
            "unresolved_mentions": [["COREF_SKIPPED_QUOTA"] for _ in range(len(missing))],
        })], ignore_index=True)

    if quota_stop:
        print("[INFO] Coref dung som vi quota; chay lai cell nay sau khi quota reset de hoan tat.")
    return result

extraction_source = chunks_df.head(EXTRACTION_MAX_CHUNKS).copy()
coref_df = cached_df("coref_df", lambda: run_coref(extraction_source))
extraction_source = extraction_source.merge(coref_df, on="chunk_id", how="left")

_failed = sum(
    1 for m in coref_df["unresolved_mentions"]
    if isinstance(m, list) and "COREF_BATCH_FAILED" in m
)
print(f"Coref: {len(coref_df)} chunks, {_failed} chunk thuoc batch that bai.")
if _failed == len(coref_df):
    raise RuntimeError("COREF FAILED: tat ca batch deu loi -> kiem tra GROQ_API_KEY/GROQ_MODEL.")


[CACHE HIT] coref_df: 200 dong (xoa .cache\coref_df.pkl de tinh lai).
Coref: 200 chunks, 5 chunk thuoc batch that bai.


# PHẦN 2 — TRIPLE EXTRACTION & NEO4J BULK INSERT (15–45')

## Graph schema
**Nodes:** `Company`, `Person`, `Technology` + base label `Entity`.

**Relations:** `ACQUIRED`, `DEVELOPED`, `INVESTED_IN`, `FOUNDED`, `WORKED_AT`, `PARTNERED_WITH`, `USES`, `LEADS`.

**Mỗi edge bắt buộc:** `source_chunk_id`, `published_date`; khuyến nghị thêm `evidence`, `confidence`.

> Relation type phải qua allowlist trước khi ghép vào Cypher.

In [9]:
#@title 2.1 — NER + RE extraction
ALLOWED_NODE_TYPES = {"Company", "Person", "Technology"}
ALLOWED_RELATIONS = {
    "ACQUIRED", "DEVELOPED", "INVESTED_IN", "FOUNDED",
    "WORKED_AT", "PARTNERED_WITH", "USES", "LEADS"
}

EXTRACT_SYSTEM = f"""
Extract a high-precision knowledge graph from tech-news text.
Allowed node types: {sorted(ALLOWED_NODE_TYPES)}
Allowed relations: {sorted(ALLOWED_RELATIONS)}
Use only explicitly supported facts. Prefer precision over recall.
Every relation needs short evidence. Return strict JSON only.
""".strip()

def extract_batch(batch_df):
    payload = [{
        "chunk_id": r.chunk_id,
        "published_date": r.published_date,
        "text": getattr(r, "resolved_text", None) or r.text,
    } for r in batch_df.itertuples(index=False)]

    prompt = f"""
Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "relations": [
        {{
          "source": "...",
          "source_type": "Company|Person|Technology",
          "relation": "ALLOWED_RELATION",
          "target": "...",
          "target_type": "Company|Person|Technology",
          "evidence": "...",
          "confidence": 0.0
        }}
      ]
    }}
  ]
}}

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()
    return groq_json(EXTRACT_SYSTEM, prompt)

EXTRACT_CKPT = CACHE_DIR / "triples_partial.pkl"
EXTRACT_DONE_CKPT = CACHE_DIR / "triples_done_chunks.pkl"

def _safe_float(v, default=0.0):
    try:
        return float(v)
    except (TypeError, ValueError):
        return default

def run_extraction(source_df, batch_size=4):
    """Resume duoc: checkpoint triple + danh sach chunk da xu ly sau MOI batch."""
    meta = source_df.set_index("chunk_id")["published_date"].to_dict()

    triples, errors = [], []
    done_chunks = set()
    if EXTRACT_CKPT.exists() and EXTRACT_DONE_CKPT.exists():
        triples = pd.read_pickle(EXTRACT_CKPT).to_dict("records")
        done_chunks = set(pd.read_pickle(EXTRACT_DONE_CKPT))
        print(f"[RESUME] extraction: {len(triples)} triple, {len(done_chunks)} chunk da xong.")

    todo = source_df[~source_df.chunk_id.isin(done_chunks)]

    for start in tqdm(range(0, len(todo), batch_size), desc="NER+RE"):
        batch = todo.iloc[start:start+batch_size]
        try:
            obj, _ = extract_batch(batch)
        except Exception as e:
            if "Tat ca model trong pool da het quota" in str(e):
                print(f"[QUOTA STOP] Dung extraction tai batch {start}, giu checkpoint.")
                break
            errors.append({"start": start, "error": str(e)})
            continue

        done_chunks.update(batch.chunk_id.tolist())

        items = obj.get("items", []) if isinstance(obj, dict) else []
        if not isinstance(items, list):
            items = []
        for item in items:
            # Model nho trong pool doi khi tra ve chuoi thay vi object -> bo qua, khong crash
            if not isinstance(item, dict):
                errors.append({"start": start, "error": f"MALFORMED_ITEM: {str(item)[:120]}"})
                continue
            cid = item.get("chunk_id")
            if cid not in meta:
                continue
            rels = item.get("relations", [])
            if not isinstance(rels, list):
                continue
            for x in rels:
                if not isinstance(x, dict):
                    continue
                s, t = norm_space(x.get("source")), norm_space(x.get("target"))
                st, tt, rel = x.get("source_type"), x.get("target_type"), x.get("relation")
                if not s or not t:
                    continue
                if st not in ALLOWED_NODE_TYPES or tt not in ALLOWED_NODE_TYPES:
                    continue
                if rel not in ALLOWED_RELATIONS:
                    continue
                triples.append({
                    "source_raw": s,
                    "source_type": st,
                    "relation": rel,
                    "target_raw": t,
                    "target_type": tt,
                    "source_chunk_id": cid,
                    "published_date": meta[cid] or "",
                    "evidence": norm_space(x.get("evidence")),
                    "confidence": _safe_float(x.get("confidence")),
                })

        pd.DataFrame(triples).to_pickle(EXTRACT_CKPT)
        pd.Series(sorted(done_chunks)).to_pickle(EXTRACT_DONE_CKPT)

    return pd.DataFrame(triples), pd.DataFrame(errors)

def _run_extraction_all():
    _tri, _err = run_extraction(extraction_source)
    globals()["extraction_errors_df"] = _err
    if len(_err):
        _err.to_pickle(CACHE_DIR / "extraction_errors_df.pkl")
    return _tri

raw_triples_df = cached_df("raw_triples_df", _run_extraction_all)
extraction_errors_df = globals().get("extraction_errors_df")
if extraction_errors_df is None:
    _p = CACHE_DIR / "extraction_errors_df.pkl"
    extraction_errors_df = pd.read_pickle(_p) if _p.exists() else pd.DataFrame()

print(f"Trich xuat duoc {len(raw_triples_df)} triple tu {len(extraction_source)} chunk.")
if len(extraction_errors_df):
    print(f"CANH BAO: {len(extraction_errors_df)} batch loi. Vi du:")
    print(extraction_errors_df.head(3).to_string())
if raw_triples_df.empty:
    raise RuntimeError(
        "EXTRACTION FAILED: 0 triple. Kiem tra GROQ_API_KEY/GROQ_MODEL "
        "hoac schema allowlist qua chat so voi du lieu."
    )
display(raw_triples_df.head())

[CACHE HIT] raw_triples_df: 64 dong (xoa .cache\raw_triples_df.pkl de tinh lai).
Trich xuat duoc 64 triple tu 200 chunk.
CANH BAO: 2 batch loi. Vi du:
   start                                                                                                                                                                                                                                error
0     36  Error code: 400 - {'error': {'message': "Failed to validate JSON. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'json_validate_failed', 'failed_generation': ''}}
1     40  Error code: 400 - {'error': {'message': "Failed to validate JSON. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'json_validate_failed', 'failed_generation': ''}}


,source_raw,source_type,relation,target_raw,target_type,source_chunk_id,published_date,evidence,confidence
0,Crexendo,Company,DEVELOPED,VIP Business Communications Platform,Technology,391d8c6c09f3574ace59::c0000,2023-04-19,Crexendo's VIP Business Communications Platform,1.0
1,Fidelity National Information Services,Company,ACQUIRED,Worldpay,Company,e932445090da75b8ccf6::c0000,2022-12-15,acquisition of Cincinnati-based Worldpay,1.0
2,Synopsys,Company,PARTNERED_WITH,TSMC,Company,4b907bbb535472c4877e::c0000,2023-05-17,Synopsys collaboration with TSMC,1.0
3,Leo Howell,Person,WORKED_AT,Institute,Company,e2d62731f2691c8351bb::c0000,2023-06-12,Leo Howell will join the president’s cabinet as the interim vice president for Information Technology and chief info...,1.0
4,Market Express,Company,USES,Just Walk Out technology,Technology,72ff44ac6bc220df7d85::c0000,2022-12-15,powered by Amazon’s Just Walk Out technology,1.0


## 2.2 — Entity Resolution bằng Vector Similarity

Pipeline:
1. Manual aliases cho ticker/tên rất phổ biến.
2. Embedding ANN candidate.
3. Lexical guard để giảm false merge.
4. Xuất audit table.

### 🎯 AI Coding Agent Challenge B
Cải tiến guard cho:
- ticker,
- suffix `Inc./Corp./Ltd.`,
- product chứa company name,
- người trùng họ/tên gần giống.

In [10]:
#@title 2.2 — Entity resolution
CORP_SUFFIXES = {"inc","incorporated","corp","corporation","ltd","limited","llc","plc","co","company"}
MANUAL_ALIASES = {
    "msft": "Microsoft",
    "microsoft corp": "Microsoft",
    "microsoft corporation": "Microsoft",
    "goog": "Google",
    "googl": "Google",
    "google llc": "Google",
    "meta platforms": "Meta",
    "meta platforms inc": "Meta",
    "aapl": "Apple",
    "apple inc": "Apple",
}

def norm_entity(name):
    s = unicodedata.normalize("NFKC", norm_space(name)).lower()
    s = re.sub(r"[^\w\s\-\.]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def strip_suffix(name):
    toks = norm_entity(name).replace(".", "").split()
    while toks and toks[-1] in CORP_SUFFIXES:
        toks.pop()
    return " ".join(toks)

def merge_guard(a, b):
    na, nb = strip_suffix(a), strip_suffix(b)
    if na == nb:
        return True
    return SequenceMatcher(None, na, nb).ratio() >= 0.72

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedder = None

def get_embedder():
    global embedder
    if embedder is None:
        embedder = SentenceTransformer(EMBED_MODEL)
    return embedder

class UF:
    def __init__(self, n):
        self.p = list(range(n))
    def find(self, x):
        if self.p[x] != x:
            self.p[x] = self.find(self.p[x])
        return self.p[x]
    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a != b:
            self.p[b] = a

def build_resolution_map(raw_triples_df, threshold=0.90, top_k=5):
    mentions = []
    for r in raw_triples_df.itertuples(index=False):
        mentions += [(r.source_type, r.source_raw), (r.target_type, r.target_raw)]

    counts = Counter((t, norm_entity(n)) for t, n in mentions)
    display_name = {}
    for t, n in mentions:
        display_name.setdefault((t, norm_entity(n)), n)

    mapping, audit = {}, []

    for key in counts:
        t, norm = key
        if norm in MANUAL_ALIASES:
            mapping[key] = MANUAL_ALIASES[norm]
            audit.append({
                "type": t, "left": display_name[key],
                "right": MANUAL_ALIASES[norm],
                "similarity": 1.0, "decision": "MERGE_MANUAL"
            })

    for typ in sorted(ALLOWED_NODE_TYPES):
        keys = [k for k in counts if k[0] == typ and k not in mapping]
        if not keys:
            continue
        names = [display_name[k] for k in keys]
        vecs = get_embedder().encode(
            names, batch_size=128, show_progress_bar=False,
            normalize_embeddings=True
        ).astype("float32")

        index = faiss.IndexFlatIP(vecs.shape[1])
        index.add(vecs)
        sims, nbrs = index.search(vecs, min(top_k, len(names)))
        uf = UF(len(names))

        for i in range(len(names)):
            for score, j in zip(sims[i], nbrs[i]):
                if j < 0 or i >= j or float(score) < threshold:
                    continue
                ok = merge_guard(names[i], names[j])
                audit.append({
                    "type": typ, "left": names[i], "right": names[j],
                    "similarity": float(score),
                    "decision": "MERGE_VECTOR" if ok else "REJECT_GUARD"
                })
                if ok:
                    uf.union(i, j)

        groups = defaultdict(list)
        for i in range(len(names)):
            groups[uf.find(i)].append(i)

        for idxs in groups.values():
            best = sorted(
                idxs,
                key=lambda i: (-counts[keys[i]], len(names[i]), names[i].lower())
            )[0]
            canonical = names[best]
            for i in idxs:
                mapping[keys[i]] = canonical

    for key in counts:
        mapping.setdefault(key, display_name[key])

    return mapping, pd.DataFrame(audit)

def canonicalize_triples(raw_df, mapping):
    df = raw_df.copy()
    def canon(name, typ):
        n = norm_entity(name)
        return mapping.get((typ, n), MANUAL_ALIASES.get(n, name))

    df["source_name"] = [canon(n,t) for n,t in zip(df.source_raw, df.source_type)]
    df["target_name"] = [canon(n,t) for n,t in zip(df.target_raw, df.target_type)]
    df["source_name_norm"] = df.source_name.map(norm_entity)
    df["target_name_norm"] = df.target_name.map(norm_entity)
    df["source_id"] = [sha1(f"{t}:{n}")[:24] for t,n in zip(df.source_type, df.source_name_norm)]
    df["target_id"] = [sha1(f"{t}:{n}")[:24] for t,n in zip(df.target_type, df.target_name_norm)]
    return df[df.source_id != df.target_id].reset_index(drop=True)

# Nguong san xuat khuyen nghi la 0.90 (xem lab_report.md muc 1.2). Voi corpus nho
# (200 chunk demo, 108 entity mention) khong co cap nao dat >= 0.90 (max thuc te = 0.644),
# nen KHONG co audit row nao duoc sinh ra o nguong do. Ha xuong 0.45 CHI DE DEMO co che
# guard tren du lieu that; khong anh huong ket qua canonical hoa vi moi cap deu bi REJECT
# o ca hai nguong (merge_guard khong bao gio tra True voi corpus nay).
entity_map, entity_resolution_audit_df = build_resolution_map(raw_triples_df, threshold=0.45)
print(f"[DEMO threshold=0.45] {len(entity_resolution_audit_df)} audit rows sinh ra "
      f"(nguong san xuat 0.90 se cho 0 rows voi corpus nho nay).")
triples_df = canonicalize_triples(raw_triples_df, entity_map)
display(entity_resolution_audit_df.head(20))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1740.75it/s]

[DEMO threshold=0.45] 22 audit rows sinh ra (nguong san xuat 0.90 se cho 0 rows voi corpus nho nay).


,type,left,right,similarity,decision
0,Company,Institute,Ivy Tech Community College,0.524012,REJECT_GUARD
1,Company,Amazon,Apple,0.485086,REJECT_GUARD
2,Company,Amazon,Samsung,0.483815,REJECT_GUARD
3,Company,Amazon,Google,0.458060,REJECT_GUARD
4,Company,Aqara,AYYA,0.618492,REJECT_GUARD
5,Company,Samsung,Apple,0.575256,REJECT_GUARD
6,Company,Apple,Microsoft,0.492803,REJECT_GUARD
7,Company,Future Technology Group,Jobs for the Future,0.562230,REJECT_GUARD
8,Company,NPR,Google,0.510892,REJECT_GUARD
9,Person,Azarzar,Mazhar Mohammad,0.450918,REJECT_GUARD


In [11]:
#@title 2.3 — Node table + UNWIND bulk insert
def build_nodes(triples_df):
    rows = []
    for r in triples_df.itertuples(index=False):
        rows += [
            {"id":r.source_id,"name":r.source_name,"name_norm":r.source_name_norm,"type":r.source_type,"alias":r.source_raw},
            {"id":r.target_id,"name":r.target_name,"name_norm":r.target_name_norm,"type":r.target_type,"alias":r.target_raw},
        ]
    tmp = pd.DataFrame(rows)
    if tmp.empty:
        return tmp

    out = []
    for (node_id,name,name_norm,typ), g in tmp.groupby(["id","name","name_norm","type"]):
        aliases = sorted(set(g["alias"].map(norm_space)))
        out.append({
            "id":node_id, "name":name, "name_norm":name_norm, "type":typ,
            "aliases":aliases,
            "aliases_norm":sorted(set(norm_entity(x) for x in aliases))
        })
    return pd.DataFrame(out)

def batches(records, size=1000):
    for i in range(0, len(records), size):
        yield records[i:i+size]

def bulk_insert_nodes(nodes_df, batch_size=1000):
    for typ in sorted(ALLOWED_NODE_TYPES):
        part = nodes_df[nodes_df.type == typ]
        if part.empty:
            continue
        query = f"""
        UNWIND $rows AS row
        MERGE (n:Entity {{id: row.id}})
        SET n:{typ},
            n.name=row.name,
            n.name_norm=row.name_norm,
            n.entity_type=row.type,
            n.aliases=row.aliases,
            n.aliases_norm=row.aliases_norm
        """
        for b in batches(part.to_dict("records"), batch_size):
            run_cypher(query, rows=b)

def bulk_insert_edges(triples_df, batch_size=1000):
    required = {"source_chunk_id","published_date"}
    if not required.issubset(triples_df.columns):
        raise ValueError("Missing edge provenance.")

    for rel in sorted(ALLOWED_RELATIONS):
        part = triples_df[triples_df.relation == rel]
        if part.empty:
            continue

        query = f"""
        UNWIND $rows AS row
        MATCH (s:Entity {{id: row.source_id}})
        MATCH (t:Entity {{id: row.target_id}})
        MERGE (s)-[r:{rel} {{source_chunk_id: row.source_chunk_id}}]->(t)
        SET r.published_date=row.published_date,
            r.evidence=row.evidence,
            r.confidence=row.confidence
        """

        cols = ["source_id","target_id","source_chunk_id","published_date","evidence","confidence"]
        for b in batches(part[cols].to_dict("records"), batch_size):
            run_cypher(query, rows=b)

nodes_df = build_nodes(triples_df)
bulk_insert_nodes(nodes_df)
bulk_insert_edges(triples_df)

In [12]:
#@title 2.4 — Sanity checks
def graph_checks():
    invalid = run_cypher("""
    MATCH ()-[r]->()
    WHERE r.source_chunk_id IS NULL OR r.published_date IS NULL
    RETURN count(r) AS n
    """)[0]["n"]

    counts = {
        "nodes": run_cypher("MATCH (n:Entity) RETURN count(n) AS n")[0]["n"],
        "edges": run_cypher("MATCH ()-[r]->() RETURN count(r) AS n")[0]["n"],
        "invalid_provenance_edges": invalid,
    }
    print(counts)
    assert invalid == 0

    top = pd.DataFrame(run_cypher("""
    MATCH (n:Entity)
    OPTIONAL MATCH (n)-[r]-()
    WITH n, count(r) AS degree
    RETURN n.id AS id, n.name AS name, n.entity_type AS type, degree
    ORDER BY degree DESC LIMIT 15
    """))
    display(top)
    return counts, top

graph_counts, top_degree_df = graph_checks()

{'nodes': 135, 'edges': 93, 'invalid_provenance_edges': 0}


,id,name,type,degree
0,707ea356c1a6dfcbe0abad36,SigenStor,Company,6
1,6b8d3ec5d642d544e137e070,Future Technology Group,Company,4
2,988a195c3c19b74e025a1af6,SC Ventures,Company,3
3,ce0b6a8e50a6a58526e4c85f,Aeris,Company,3
4,fb0f4df56fab164ec48722f0,Microsoft,Company,3
5,0daf004a34fe4388fe295005,Dalet,Company,3
6,e6c3db5c28c9a15bfafd2e04,Apple,Company,3
7,68b3544368862fc263941c8d,Amazon,Company,3
8,0e132222d1315530d6efca52,Google,Company,3
9,8568cefc2b16b1699bc0d89f,Aqara,Company,2


# PHẦN 3 — FLAT RAG & HYBRID GRAPHRAG (45–75')

## Flat RAG baseline
Dùng cùng embedding/generator để comparison tập trung vào retrieval architecture.

In [13]:
#@title 3.1 — Flat RAG
flat_index = None
flat_store = None
entity_match_vectors = None
entity_match_store = None

def build_flat_index(chunks_df):
    global flat_index, flat_store
    vecs = get_embedder().encode(
        chunks_df.text.fillna("").tolist(),
        batch_size=128, show_progress_bar=True,
        normalize_embeddings=True
    ).astype("float32")

    flat_index = faiss.IndexFlatIP(vecs.shape[1])
    flat_index.add(vecs)
    flat_store = chunks_df.reset_index(drop=True).copy()
    print("Flat vectors:", flat_index.ntotal)

def retrieve_flat_context(query, k=6):
    qv = get_embedder().encode(
        [query], normalize_embeddings=True, show_progress_bar=False
    ).astype("float32")
    scores, ids = flat_index.search(qv, min(k, flat_index.ntotal))

    rows = []
    for score, idx in zip(scores[0], ids[0]):
        if idx < 0:
            continue
        r = flat_store.iloc[int(idx)]
        rows.append({
            "score":float(score), "chunk_id":r.chunk_id,
            "published_date":r.published_date, "text":r.text
        })

    df = pd.DataFrame(rows)
    context = "\n\n".join(
        f"[chunk_id={r.chunk_id} | date={r.published_date} | score={r.score:.3f}]\n{r.text}"
        for r in df.itertuples(index=False)
    )
    return context, df

build_flat_index(chunks_df)

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Batches:   8%|▊         | 1/12 [00:09<01:41,  9.22s/it]

Batches:  17%|█▋        | 2/12 [00:11<00:52,  5.28s/it]

Batches:  25%|██▌       | 3/12 [00:14<00:39,  4.34s/it]

Batches:  33%|███▎      | 4/12 [00:17<00:28,  3.60s/it]

...

Batches:  83%|████████▎ | 10/12 [00:27<00:03,  1.80s/it]

Batches:  92%|█████████▏| 11/12 [00:28<00:01,  1.67s/it]

Batches: 100%|██████████| 12/12 [00:29<00:00,  1.34s/it]

Batches: 100%|██████████| 12/12 [00:29<00:00,  2.42s/it]

Flat vectors: 1501


## Graph retrieval flow
1. LLM trích seed entities.
2. Match seed trong Neo4j; fuzzy fallback bằng embedding.
3. BFS tối đa `max_hops`.
4. Nếu node degree > 100 → chỉ lấy tối đa 50 edge mới nhất.
5. Global edge cap để tránh context explosion.
6. Textualize subgraph có provenance.

In [14]:
#@title 3.2 — Seed matching
SEED_SYSTEM = """
Extract useful seed entities for graph retrieval.
Allowed types: Company, Person, Technology.
Do not answer the question. Return strict JSON only.
""".strip()

def extract_seeds(query):
    obj, _ = groq_json(SEED_SYSTEM, f"""
Question: {query}
Return {{"seeds":[{{"name":"...","type":"Company|Person|Technology|null"}}]}}
""")
    return [
        {"name":norm_space(x.get("name")),
         "type":x.get("type") if x.get("type") in ALLOWED_NODE_TYPES else None}
        for x in obj.get("seeds", [])
        if norm_space(x.get("name"))
    ]

def build_entity_matcher(nodes_df):
    global entity_match_vectors, entity_match_store
    entity_match_store = nodes_df.reset_index(drop=True).copy()
    entity_match_vectors = get_embedder().encode(
        entity_match_store.name.tolist(),
        batch_size=128, show_progress_bar=False,
        normalize_embeddings=True
    ).astype("float32")

def match_seeds(query, fuzzy_threshold=0.66):
    matched = []
    for seed in extract_seeds(query):
        exact = run_cypher("""
        MATCH (n:Entity)
        WHERE (n.name_norm=$name OR $name IN coalesce(n.aliases_norm,[]))
          AND ($typ IS NULL OR n.entity_type=$typ)
        RETURN n.id AS id, n.name AS name, n.entity_type AS type
        LIMIT 5
        """, name=norm_entity(seed["name"]), typ=seed["type"])

        if exact:
            matched += exact
            continue

        if entity_match_vectors is None:
            continue

        mask = np.ones(len(entity_match_store), dtype=bool)
        if seed["type"]:
            mask = entity_match_store.type.eq(seed["type"]).to_numpy()
        idxs = np.flatnonzero(mask)
        if not len(idxs):
            continue

        qv = get_embedder().encode(
            [seed["name"]], normalize_embeddings=True, show_progress_bar=False
        ).astype("float32")[0]
        sims = entity_match_vectors[idxs] @ qv
        j = int(np.argmax(sims))
        if float(sims[j]) >= fuzzy_threshold:
            r = entity_match_store.iloc[int(idxs[j])]
            matched.append({"id":r.id,"name":r.name,"type":r.type})

    return list({x["id"]: x for x in matched}.values())

build_entity_matcher(nodes_df)

In [15]:
#@title 3.3 — Graph traversal + super-node mitigation
SUPER_NODE_DEGREE = 100
SUPER_NODE_EDGE_CAP = 50
GLOBAL_EDGE_CAP = 250
MAX_GRAPH_CONTEXT_CHARS = 14000

def node_degree(node_id):
    return int(run_cypher("""
    MATCH (n:Entity {id:$id})
    OPTIONAL MATCH (n)-[r]-()
    RETURN count(r) AS degree
    """, id=node_id)[0]["degree"])

def recent_edges(node_id, limit):
    return run_cypher("""
    MATCH (n:Entity {id:$id})
    MATCH (n)-[r]-(m:Entity)
    RETURN
      startNode(r).id AS source_id,
      startNode(r).name AS source_name,
      startNode(r).entity_type AS source_type,
      type(r) AS relation,
      endNode(r).id AS target_id,
      endNode(r).name AS target_name,
      endNode(r).entity_type AS target_type,
      r.source_chunk_id AS source_chunk_id,
      r.published_date AS published_date,
      r.evidence AS evidence,
      m.id AS neighbor_id
    ORDER BY coalesce(r.published_date,'') DESC
    LIMIT $limit
    """, id=node_id, limit=int(limit))

def textualize(edges):
    edges = sorted(edges, key=lambda e:e.get("published_date") or "", reverse=True)
    lines, used = [], 0
    for e in edges:
        line = (
            f"{e['source_name']} [{e['source_type']}] -{e['relation']}-> "
            f"{e['target_name']} [{e['target_type']}] "
            f"| date={e.get('published_date') or 'unknown'} "
            f"| chunk={e.get('source_chunk_id') or 'unknown'}"
        )
        if e.get("evidence"):
            line += f" | evidence={norm_space(e['evidence'])}"
        if used + len(line) + 1 > MAX_GRAPH_CONTEXT_CHARS:
            break
        lines.append(line)
        used += len(line) + 1
    return "\n".join(lines)

def retrieve_graph_context(query, max_hops=2, edge_limit=50, return_debug=False):
    seeds = match_seeds(query)
    if not seeds:
        out = {"context":"","edges":pd.DataFrame(),
               "diagnostics":{"reason":"NO_SEED","supernode_events":[]}}
        return out if return_debug else ""

    frontier = deque((x["id"],0) for x in seeds)
    expanded, seen_edges, collected = set(), set(), []
    supernode_events = []

    while frontier and len(collected) < GLOBAL_EDGE_CAP:
        node_id, hop = frontier.popleft()
        if node_id in expanded or hop >= max_hops:
            continue
        expanded.add(node_id)

        degree = node_degree(node_id)
        limit = int(edge_limit)
        if degree > SUPER_NODE_DEGREE:
            limit = min(limit, SUPER_NODE_EDGE_CAP)
            supernode_events.append({"node_id":node_id,"degree":degree,"limit":limit})

        for e in recent_edges(node_id, limit):
            key = (e["source_id"],e["relation"],e["target_id"],e["source_chunk_id"])
            if key in seen_edges:
                continue
            seen_edges.add(key)
            collected.append(e)
            if len(collected) >= GLOBAL_EDGE_CAP:
                break

            nb = e.get("neighbor_id")
            if nb and nb not in expanded and hop + 1 < max_hops:
                frontier.append((nb, hop+1))

    out = {
        "context": textualize(collected),
        "edges": pd.DataFrame(collected),
        "diagnostics": {
            "matched_seeds": seeds,
            "expanded_nodes": len(expanded),
            "collected_edges": len(collected),
            "supernode_events": supernode_events,
        }
    }
    return out if return_debug else out["context"]

In [16]:
#@title 3.4 — Flat answer vs Hybrid GraphRAG answer
ANSWER_SYSTEM = """
Answer only from supplied context.
Be concise but complete. Do not invent facts.
Cite provenance inline as [chunk_id=...] whenever possible.
If evidence is insufficient or conflicting, say so.
""".strip()

def generate_answer(question, context):
    prompt = f"QUESTION:\n{question}\n\nCONTEXT:\n{context}\n\nANSWER:"
    t0 = time.perf_counter()
    text, usage = groq_chat(
        [{"role":"system","content":ANSWER_SYSTEM},
         {"role":"user","content":prompt}]
    )
    return {
        "answer": text.strip(),
        "latency_s": time.perf_counter()-t0,
        "total_tokens": usage.get("total_tokens"),
    }

def answer_flat_rag(question):
    context, retrieved = retrieve_flat_context(question, k=6)
    out = generate_answer(question, context)
    out.update({"context":context,"retrieved":retrieved})
    return out

def answer_graph_rag(question):
    g = retrieve_graph_context(question, max_hops=2, edge_limit=50, return_debug=True)
    vctx, vdocs = retrieve_flat_context(question, k=4)
    context = f"=== GRAPH ===\n{g['context']}\n\n=== VECTOR ===\n{vctx}"
    out = generate_answer(question, context)
    out.update({"context":context,"graph_debug":g,"vector_docs":vdocs})
    return out

# PHẦN 4 — GOLDEN DATASET & LLM-AS-A-JUDGE (75–105')

## Golden schema
`id`, `group`, `question`, `reference_answer`, optional `reference_evidence`.

Notebook có 5 câu starter. Các câu phụ thuộc data dump phải điền gold answer thật trước final evaluation.

In [17]:
#@title 4.1 — 5 câu Golden starter
GOLDEN_PATH = str(Path(REPO_DIR)/"data"/"golden_dataset.csv") if "REPO_DIR" in dir() and Path(REPO_DIR).exists() else ("/content/golden_dataset.csv" if IN_COLAB else "data/golden_dataset.csv")

starter_golden = pd.DataFrame([
    {
        "id":"G01","group":"factoid",
        "question":"Who was the CEO of Hugging Face in 2023?",
        "reference_answer":"Clément Delangue",
        "reference_evidence":"Validate against instructor dump."
    },
    {
        "id":"G02","group":"multi-hop",
        "question":"Which startups were founded by former Microsoft employees and later received investment from Google?",
        "reference_answer":"",
        "reference_evidence":"TO_BE_FILLED_FROM_DATASET"
    },
    {
        "id":"G03","group":"cross-doc",
        "question":"Compare the direction of AI-related investments by Meta and Apple during 2023 using evidence from multiple articles.",
        "reference_answer":"",
        "reference_evidence":"TO_BE_FILLED_FROM_DATASET"
    },
    {
        "id":"G04","group":"multi-hop",
        "question":"Find a company invested in by a major technology company that also developed a named AI technology; identify both relations and dates.",
        "reference_answer":"",
        "reference_evidence":"TO_BE_FILLED_FROM_DATASET"
    },
    {
        "id":"G05","group":"cross-doc",
        "question":"Identify one technology connected to the same company in at least two news chunks and summarize how the relationship changed over time.",
        "reference_answer":"",
        "reference_evidence":"TO_BE_FILLED_FROM_DATASET"
    },
])

golden_df = pd.read_csv(GOLDEN_PATH) if Path(GOLDEN_PATH).exists() else starter_golden.copy()
display(golden_df)

def validate_golden(df, require_answers=True):
    required = {"id","group","question","reference_answer"}
    if not required.issubset(df.columns):
        raise ValueError(f"Missing columns: {required-set(df.columns)}")
    if require_answers and df.reference_answer.fillna("").str.strip().eq("").any():
        display(df[df.reference_answer.fillna("").str.strip().eq("")][["id","question"]])
        raise ValueError("Điền reference_answer trước final evaluation.")
    print("✅ Golden Dataset valid.")

,id,group,question,reference_answer,reference_evidence
0,G5000-26,multi-hop,"What external technology provider is named inside Amazon's July AI-service expansion, and what other new AI capabili...",Amazon's AI-service story names access to technology from Cohere. It also mentions a program for building more conve...,row 2532 (2023-07-26 20:19:00): Exclusive: Amazon has drawn thousands to try its AI service competing with Microsoft...
1,G5000-27,cross-doc,How should the graph reconcile the statement that AMD powers multiple cloud services with the later Reuters report a...,The June 1 investment article broadly says AMD powers multiple cloud services through its chips. The June 14 Reuters...,row 3357 (2023-06-01 13:16:00): 3 Best Cloud Stocks to Buy in June | row 2905 (2023-06-14 08:47:00): Exclusive: Amaz...
2,G5000-28,multi-hop,"Which model providers are connected to Google Cloud Next '23 in the selected data, and which models are associated w...",Meta is connected via Llama 2 and Code Llama; the Technology Innovation Institute is connected via Falcon LLM; Anthr...,row 3395 (2023-08-29 18:07:00): Google Cloud Kicks Off Next '23 with a New Way to Cloud
3,G5000-29,cross-doc,How did participation in White House AI commitments broaden from July to September 2023 according to the selected re...,"The July report names seven companies including Google, Meta, and OpenAI as making voluntary AI commitments. The Sep...",row 3380 (2023-07-21 13:01:00): The White House and big tech companies release commitments on managing AI | row 3330...
4,G5000-30,multi-hop,"Meta appears in two different AI contexts in the selected data. What are they, and what distinct relation should the...","At Google Cloud Next, Meta is the provider/source of Llama 2 and Code Llama models made available on Google Cloud. S...",row 3395 (2023-08-29 18:07:00): Google Cloud Kicks Off Next '23 with a New Way to Cloud | row 3380 (2023-07-21 13:01...
5,G5000-31,multi-hop,"Order OpenAI's ecosystem moves from March through July 2023 using the selected sources: plug-ins, open-source model ...",March: ChatGPT gained support for about a dozen application plug-ins. May: OpenAI was reported to be preparing a new...,row 3938 (2023-03-24 15:58:00): OpenAI's ChatGPT gets support for a dozen application plug-ins | row 946 (2023-05-15...
6,G5000-32,cross-doc,What is the difference between OpenAI's March plug-in development and its June reported app-store plan?,The March story describes ChatGPT gaining support for application plug-ins so companies can expose product functiona...,row 3938 (2023-03-24 15:58:00): OpenAI's ChatGPT gets support for a dozen application plug-ins | row 2449 (2023-06-2...
7,G5000-33,cross-doc,"Which July OpenAI-related event is a content/technology collaboration, and which July event is a voluntary governanc...",The AP–OpenAI agreement is a collaboration to share access to select news content and technology for generative-AI u...,row 366 (2023-07-13 00:00:00): AP Open AI agree to share select news content and technology in new collaboration | r...
8,G5000-34,multi-hop,Compare how Google Cloud and Amazon expanded their AI ecosystems in the selected data. Which third-party model/techn...,"Google Cloud announced models from Meta (Llama 2 and Code Llama), the Technology Innovation Institute (Falcon LLM), ...",row 3395 (2023-08-29 18:07:00): Google Cloud Kicks Off Next '23 with a New Way to Cloud | row 2532 (2023-07-26 20:19...
9,G5000-35,cross-doc,Contrast AWS's AMD-chip posture with HPE's AI-cloud posture. Which is a tentative hardware sourcing decision and whi...,"AWS was only considering AMD's new AI chips, with no final decision. HPE said it would offer a cloud computing servi...",row 2905 (2023-06-14 08:47:00): Exclusive: Amazon's cloud unit is considering AMD's new AI chips | row 3289 (2023-06...


In [18]:
#@title 4.2 — LLM-as-a-Judge
JUDGE_SYSTEM = """
You are a strict evaluator of RAG answers.
Score 1-5:
- comprehensiveness
- faithfulness to supplied candidate context
- multi_hop_reasoning accuracy
Use the reference answer as correctness anchor.
Return strict JSON only.
""".strip()

def judge_json(system, user):
    if not JUDGE_MODEL:
        raise RuntimeError("Thiếu JUDGE_MODEL.")

    if JUDGE_PROVIDER == "groq":
        return groq_json(system, user, model=JUDGE_MODEL)[0]

    if JUDGE_PROVIDER == "openai":
        if not OPENAI_API_KEY:
            raise RuntimeError("Thiếu OPENAI_API_KEY.")
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)
        resp = client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[{"role":"system","content":system},
                      {"role":"user","content":user}],
            temperature=0.0,
            response_format={"type":"json_object"}
        )
        return parse_json_object(resp.choices[0].message.content)

    raise ValueError("JUDGE_PROVIDER must be openai or groq.")

def judge_answer(question, reference, answer, context):
    prompt = f"""
QUESTION:
{question}

REFERENCE:
{reference}

CANDIDATE:
{answer}

CANDIDATE CONTEXT:
{context[:18000]}

Return:
{{
 "comprehensiveness":1,
 "faithfulness":1,
 "multi_hop_reasoning":1,
 "rationale":"2-5 sentences"
}}
"""
    obj = judge_json(JUDGE_SYSTEM, prompt)
    out = {}
    for k in ["comprehensiveness","faithfulness","multi_hop_reasoning"]:
        out[k] = max(1, min(5, int(obj.get(k,1))))
    out["rationale"] = norm_space(obj.get("rationale"))
    return out

In [19]:
#@title 4.3 — Evaluation runner + checkpoint
CHECKPOINT = str(Path(REPO_DIR)/"outputs"/"graphrag_eval_checkpoint.csv") if "REPO_DIR" in dir() and Path(REPO_DIR).exists() else ("/content/graphrag_eval_checkpoint.csv" if IN_COLAB else "outputs/graphrag_eval_checkpoint.csv")

def run_evaluation(golden_df):
    rows = []
    done_ids = set()
    if Path(CHECKPOINT).exists():
        prev = pd.read_csv(CHECKPOINT)
        rows = prev.to_dict("records")
        done_ids = set(prev["id"])
        print(f"[RESUME] evaluation: {len(done_ids)}/{len(golden_df)} cau da xong.")

    todo = golden_df[~golden_df["id"].isin(done_ids)]
    for q in tqdm(todo.itertuples(index=False), total=len(todo), desc="Evaluation"):
        try:
            flat = answer_flat_rag(q.question)
            graph = answer_graph_rag(q.question)
        except RuntimeError as e:
            if "het quota" in str(e) or "rate_limit" in str(e):
                print(f"[QUOTA STOP] Dung tai cau {q.id}, giu checkpoint ({len(rows)} cau da xong).")
                break
            raise

        jf = judge_answer(q.question, q.reference_answer, flat["answer"], flat["context"])
        jg = judge_answer(q.question, q.reference_answer, graph["answer"], graph["context"])

        rows.append({
            "id":q.id, "group":q.group, "question":q.question,
            "reference_answer":q.reference_answer,
            "flat_answer":flat["answer"], "graph_answer":graph["answer"],
            "flat_comprehensiveness":jf["comprehensiveness"],
            "graph_comprehensiveness":jg["comprehensiveness"],
            "flat_faithfulness":jf["faithfulness"],
            "graph_faithfulness":jg["faithfulness"],
            "flat_multi_hop_reasoning":jf["multi_hop_reasoning"],
            "graph_multi_hop_reasoning":jg["multi_hop_reasoning"],
            "flat_latency_s":flat["latency_s"],
            "graph_latency_s":graph["latency_s"],
            "flat_total_tokens":flat.get("total_tokens"),
            "graph_total_tokens":graph.get("total_tokens"),
            "flat_judge_rationale":jf["rationale"],
            "graph_judge_rationale":jg["rationale"],
            "graph_supernode_events":len(
                graph["graph_debug"]["diagnostics"].get("supernode_events",[])
            )
        })
        pd.DataFrame(rows).to_csv(CHECKPOINT, index=False)
    return pd.DataFrame(rows)

validate_golden(golden_df, require_answers=True)
eval_results_df = run_evaluation(golden_df)
display(eval_results_df)

✅ Golden Dataset valid.


Evaluation:   0%|          | 0/25 [00:00<?, ?it/s]

Evaluation:   4%|▍         | 1/25 [00:13<05:22, 13.45s/it]

Evaluation:   8%|▊         | 2/25 [00:19<03:29,  9.12s/it]

[QUOTA] Model 'openai/gpt-oss-120b' het TPD -> chuyen model ke tiep.


...

Evaluation:  92%|█████████▏| 23/25 [03:52<00:26, 13.47s/it]

Evaluation:  96%|█████████▌| 24/25 [04:02<00:12, 12.46s/it]

Evaluation: 100%|██████████| 25/25 [04:18<00:00, 13.54s/it]

Evaluation: 100%|██████████| 25/25 [04:18<00:00, 10.35s/it]

,id,group,question,reference_answer,flat_answer,graph_answer,flat_comprehensiveness,graph_comprehensiveness,flat_faithfulness,graph_faithfulness,flat_multi_hop_reasoning,graph_multi_hop_reasoning,flat_latency_s,graph_latency_s,flat_total_tokens,graph_total_tokens,flat_judge_rationale,graph_judge_rationale,graph_supernode_events
0,G5000-26,multi-hop,"What external technology provider is named inside Amazon's July AI-service expansion, and what other new AI capabili...",Amazon's AI-service story names access to technology from Cohere. It also mentions a program for building more conve...,"The provided excerpts do not contain any information about Amazon’s July AI‑service expansion, the external technolo...","The provided context does not contain any information about Amazon’s July AI‑service expansion, the external technol...",1,1,1,1,1,1,1.059989,0.789119,683,1035,"The candidate does not provide any relevant information regarding Amazon's July AI-service expansion, the external t...","The candidate does not provide any information relevant to the question about Amazon's July AI-service expansion, th...",0
1,G5000-27,cross-doc,How should the graph reconcile the statement that AMD powers multiple cloud services with the later Reuters report a...,The June 1 investment article broadly says AMD powers multiple cloud services through its chips. The June 14 Reuters...,"The provided excerpts do not contain any statements about AMD powering multiple cloud services, nor do they include ...",The provided excerpts do not contain any information about AMD powering multiple cloud services or a Reuters report ...,1,1,1,1,1,1,0.759455,0.691661,658,541,"The candidate fails to address the question by stating that the necessary information is missing, which is incorrect...",The candidate fails to address the question regarding the reconciliation of AMD's role in powering cloud services wi...,0
2,G5000-28,multi-hop,"Which model providers are connected to Google Cloud Next '23 in the selected data, and which models are associated w...",Meta is connected via Llama 2 and Code Llama; the Technology Innovation Institute is connected via Falcon LLM; Anthr...,"The provided excerpts do not contain any information about Google Cloud Next ‘23, the model providers that were conn...","I’m sorry, but the supplied context does not contain any information about Google Cloud Next ’23, its connected mode...",1,1,1,1,1,1,0.737176,1.109087,627,893,The candidate fails to provide any relevant information regarding the model providers connected to Google Cloud Next...,The candidate fails to provide any relevant information regarding the model providers connected to Google Cloud Next...,0
3,G5000-29,cross-doc,How did participation in White House AI commitments broaden from July to September 2023 according to the selected re...,"The July report names seven companies including Google, Meta, and OpenAI as making voluntary AI commitments. The Sep...","I’m sorry, but I couldn’t find any information in the supplied context that describes how participation in White Hou...","I’m sorry, but the supplied context does not contain any information about White House AI commitments or how partici...",1,1,1,1,1,1,0.648174,0.553813,631,922,The candidate fails to provide any relevant information regarding the participation in White House AI commitments fr...,The candidate response fails to address the question regarding the participation in White House AI commitments and d...,0
4,G5000-30,multi-hop,"Meta appears in two different AI contexts in the selected data. What are they, and what distinct relation should the...","At Google Cloud Next, Meta is the provider/source of Llama 2 and Code Llama models made available on Google Cloud. S...","I couldn’t locate any mention of “Meta” in the supplied chunks, so I can’t identify two distinct AI contexts or the ...","Evidence is insufficient. The provided context does not mention ""Meta"" in any AI context, nor does it describe disti...",1,1,1,1,

In [20]:
#@title 4.4 — Comparison table + export
def comparison_table(eval_df):
    metric_map = {
        "Comprehensiveness":("flat_comprehensiveness","graph_comprehensiveness"),
        "Faithfulness":("flat_faithfulness","graph_faithfulness"),
        "Multi-hop reasoning":("flat_multi_hop_reasoning","graph_multi_hop_reasoning"),
        "Latency (s)":("flat_latency_s","graph_latency_s"),
        "Token usage":("flat_total_tokens","graph_total_tokens"),
    }

    rows = []
    for group, g in eval_df.groupby("group"):
        for metric, (fc,gc) in metric_map.items():
            f = pd.to_numeric(g[fc], errors="coerce").mean()
            gr = pd.to_numeric(g[gc], errors="coerce").mean()

            if metric in {"Latency (s)","Token usage"}:
                comment = "Flat RAG thường rẻ/nhanh hơn." if f < gr else "GraphRAG không đắt hơn trong sample này."
            else:
                delta = gr - f
                if delta >= .75:
                    comment = "GraphRAG cải thiện rõ; kiểm tra rationale và provenance."
                elif delta <= -.5:
                    comment = "Flat RAG tốt hơn; graph extraction/retrieval có thể gây mất thông tin hoặc nhiễu."
                else:
                    comment = "Hai phương pháp gần nhau."

            rows.append({
                "Loại câu hỏi":group, "Metric":metric,
                "Flat RAG":round(f,3) if pd.notna(f) else np.nan,
                "GraphRAG":round(gr,3) if pd.notna(gr) else np.nan,
                "Nhận xét phân tích":comment
            })
    return pd.DataFrame(rows)

comparison_df = comparison_table(eval_results_df)
display(comparison_df)
OUT_DIR = Path(REPO_DIR)/"outputs" if "REPO_DIR" in dir() and Path(REPO_DIR).exists() else (Path("/content") if IN_COLAB else Path("outputs"))
eval_results_df.to_csv(OUT_DIR/"graphrag_eval_results.csv", index=False)
comparison_df.to_csv(OUT_DIR/"graphrag_vs_flatrag_summary.csv", index=False)

,Loại câu hỏi,Metric,Flat RAG,GraphRAG,Nhận xét phân tích
0,cross-doc,Comprehensiveness,1.000,1.000,Hai phương pháp gần nhau.
1,cross-doc,Faithfulness,1.000,1.000,Hai phương pháp gần nhau.
2,cross-doc,Multi-hop reasoning,1.000,1.000,Hai phương pháp gần nhau.
3,cross-doc,Latency (s),1.283,1.641,Flat RAG thường rẻ/nhanh hơn.
4,cross-doc,Token usage,750.727,651.727,GraphRAG không đắt hơn trong sample này.
5,factoid,Comprehensiveness,1.000,1.000,Hai phương pháp gần nhau.
6,factoid,Faithfulness,1.000,1.000,Hai phương pháp gần nhau.
7,factoid,Multi-hop reasoning,1.000,1.000,Hai phương pháp gần nhau.
8,factoid,Latency (s),1.529,2.942,Flat RAG thường rẻ/nhanh hơn.
9,factoid,Token usage,882.000,767.500,GraphRAG không đắt hơn trong sample này.


# PHẦN 5 — FAILURE-MODE CHECKS & SUBMISSION (105–120')

Bắt buộc chứng minh:
1. Edge provenance không thiếu.
2. Entity Resolution có audit.
3. Super-node degree > 100 chỉ expand tối đa 50 edge.
4. Có comparison table.

In [21]:
#@title 5.1 — Super-node check + entity audit
def test_supernode_policy(degree_threshold=None, edge_cap=SUPER_NODE_EDGE_CAP):
    """degree_threshold=None -> dung nguong san xuat SUPER_NODE_DEGREE (100).
    Truyen so nho hon de DEMO co che cap tren do thi nho, KHONG doi global
    SUPER_NODE_DEGREE nen khong anh huong retrieve_graph_context() da chay o Phan 3/4."""
    threshold = SUPER_NODE_DEGREE if degree_threshold is None else degree_threshold
    rows = run_cypher("""
    MATCH (n:Entity)-[r]-()
    WITH n, count(r) AS degree
    ORDER BY degree DESC LIMIT 1
    RETURN n.id AS id, n.name AS name, degree
    """)
    if not rows:
        print("Graph empty.")
        return

    n = rows[0]
    limit = edge_cap if n["degree"] > threshold else 1000
    edges = recent_edges(n["id"], limit)
    print(n, "fetched=", len(edges), f"(threshold={threshold})")
    if n["degree"] > threshold:
        assert len(edges) <= edge_cap
        print(f"✅ Super-node cap OK (threshold={threshold}, cap={edge_cap}).")
    else:
        print(f"Top node degree ({n['degree']}) chua vuot threshold ({threshold}) -> cap chua kich hoat.")

def show_resolution_audit(audit_df):
    if audit_df.empty:
        print("No audit rows.")
        return
    display(
        audit_df.sort_values("similarity", ascending=False).head(30)
    )
    print("High-similarity rejected pairs:")
    display(
        audit_df[audit_df.decision=="REJECT_GUARD"]
        .sort_values("similarity", ascending=False)
        .head(20)
    )

test_supernode_policy()  # nguong san xuat that: SUPER_NODE_DEGREE=100

print()
print("--- DEMO: do thi 200-chunk qua nho de bac tu nhien vuot 100, nen ha nguong ---")
print("--- xuong 3 CHI DE CHUNG MINH co che cap thuc su kich hoat & gioi han dung ---")
test_supernode_policy(degree_threshold=3)

show_resolution_audit(entity_resolution_audit_df)

{'id': '707ea356c1a6dfcbe0abad36', 'name': 'SigenStor', 'degree': 6} fetched= 6 (threshold=100)
Top node degree (6) chua vuot threshold (100) -> cap chua kich hoat.

--- DEMO: do thi 200-chunk qua nho de bac tu nhien vuot 100, nen ha nguong ---
--- xuong 3 CHI DE CHUNG MINH co che cap thuc su kich hoat & gioi han dung ---
{'id': '707ea356c1a6dfcbe0abad36', 'name': 'SigenStor', 'degree': 6} fetched= 6 (threshold=3)
✅ Super-node cap OK (threshold=3, cap=50).


,type,left,right,similarity,decision
17,Technology,high-speed fiber internet,optical fiber,0.644033,REJECT_GUARD
21,Technology,Renoworks FastTrack,Renoworks API,0.634600,REJECT_GUARD
4,Company,Aqara,AYYA,0.618492,REJECT_GUARD
5,Company,Samsung,Apple,0.575256,REJECT_GUARD
7,Company,Future Technology Group,Jobs for the Future,0.562230,REJECT_GUARD
0,Company,Institute,Ivy Tech Community College,0.524012,REJECT_GUARD
12,Person,Paul Cowan,Noel Smyth,0.514485,REJECT_GUARD
8,Company,NPR,Google,0.510892,REJECT_GUARD
15,Technology,Connected Vehicle Cloud,Saviynt cloud IAM solution,0.508019,REJECT_GUARD
16,Technology,Connected Vehicle Cloud,H2O AI Cloud,0.499335,REJECT_GUARD


High-similarity rejected pairs:


,type,left,right,similarity,decision
17,Technology,high-speed fiber internet,optical fiber,0.644033,REJECT_GUARD
21,Technology,Renoworks FastTrack,Renoworks API,0.634600,REJECT_GUARD
4,Company,Aqara,AYYA,0.618492,REJECT_GUARD
5,Company,Samsung,Apple,0.575256,REJECT_GUARD
7,Company,Future Technology Group,Jobs for the Future,0.562230,REJECT_GUARD
0,Company,Institute,Ivy Tech Community College,0.524012,REJECT_GUARD
12,Person,Paul Cowan,Noel Smyth,0.514485,REJECT_GUARD
8,Company,NPR,Google,0.510892,REJECT_GUARD
15,Technology,Connected Vehicle Cloud,Saviynt cloud IAM solution,0.508019,REJECT_GUARD
16,Technology,Connected Vehicle Cloud,H2O AI Cloud,0.499335,REJECT_GUARD


## 5.2 — Thuyết minh kỹ thuật: học viên tự điền

1. Coreference sai ở tình huống nào?
2. Entity threshold bao nhiêu, vì sao?
3. Candidate nào similarity cao nhưng không nên merge?
4. Top 3 super-node và degree?
5. Vì sao ưu tiên edge mới nhất có thể đúng/sai?
6. Flat RAG thắng nhóm nào?
7. GraphRAG thắng nhóm nào?
8. Latency/token trade-off?
9. AI Coding Agent đề xuất gì mà bạn **không dùng**, vì sao?
10. Scale 350MB: bottleneck đầu tiên là gì?

# 🎁 BONUS

## A — Low-level / High-level
Tạo local entities và high-level topics/community reports; query router chọn tầng retrieval.

## B — Global Search via Community Reports
Nếu Neo4j instance không có GDS phù hợp, fallback:
1. export edges,
2. NetworkX community detection,
3. `UNWIND` write `community_id`,
4. LLM summarize community,
5. query global trên reports.

## C — Self-Correction Graph Retrieval
- hop 2 → LLM kiểm tra context đủ chưa,
- thiếu → hop 3,
- vẫn thiếu → vector fallback,
- bắt buộc stop condition.

In [22]:
#@title Bonus — NetworkX community fallback
import networkx as nx

def build_communities(limit_edges=20000):
    edge_df = pd.DataFrame(run_cypher("""
    MATCH (a:Entity)-[r]->(b:Entity)
    RETURN a.id AS source, b.id AS target
    LIMIT $limit
    """, limit=int(limit_edges)))

    G = nx.Graph()
    G.add_edges_from(edge_df[["source","target"]].itertuples(index=False, name=None))
    communities = nx.algorithms.community.greedy_modularity_communities(G)

    rows = []
    for cid, members in enumerate(communities):
        rows += [{"id":node_id,"community_id":int(cid)} for node_id in members]

    for b in batches(rows, 1000):
        run_cypher("""
        UNWIND $rows AS row
        MATCH (n:Entity {id:row.id})
        SET n.community_id=row.community_id
        """, rows=b)

    return pd.DataFrame(rows)

# community_df = build_communities()

In [23]:
#@title Bonus — Self-correction scaffold
SUFFICIENCY_SYSTEM = """
Decide whether the supplied retrieval context is sufficient to answer the question faithfully.
Do not answer the question. Return strict JSON only.
""".strip()

def context_sufficient(question, context):
    obj, _ = groq_json(
        SUFFICIENCY_SYSTEM,
        f"""QUESTION: {question}
CONTEXT:
{context[:16000]}
Return {{"sufficient":true,"missing":"..."}}"""
    )
    return bool(obj.get("sufficient")), norm_space(obj.get("missing"))

def self_correcting_context(question):
    g2 = retrieve_graph_context(question, 2, 50, True)
    ok, missing = context_sufficient(question, g2["context"])
    if ok:
        return {"route":"hop2","context":g2["context"],"missing":""}

    g3 = retrieve_graph_context(question, 3, 50, True)
    ok, missing2 = context_sufficient(question, g3["context"])
    if ok:
        return {"route":"hop3","context":g3["context"],"missing":missing}

    flat, _ = retrieve_flat_context(question, k=8)
    return {
        "route":"hop3+vector",
        "context":f"=== GRAPH ===\n{g3['context']}\n\n=== VECTOR ===\n{flat}",
        "missing":missing2
    }

# ✅ RUBRIC

- **30% Chạy được code:** graph nạp thành công, schema đúng, xuất bảng.
- **30% Failure modes:** xử lý ít nhất 2/3 vấn đề Super-node, Entity Resolution, Coreference.
- **20% Evaluation:** chạy hết Golden Dataset, phân tích hợp lý.
- **20% Thuyết minh:** giải thích kiến trúc và cách kiểm soát AI Coding Agent.

## Submission checklist
- [ ] Neo4j connected
- [ ] Dedup/chunking đã chạy
- [ ] Coreference spot-check
- [ ] Entity resolution audit
- [ ] `UNWIND` bulk insert
- [ ] 0 edge thiếu provenance
- [ ] Flat RAG chạy
- [ ] GraphRAG chạy
- [ ] Super-node check
- [ ] Golden Dataset có gold answers thật
- [ ] Evaluation chạy hết
- [ ] Export results + summary CSV
- [ ] Thuyết minh kỹ thuật
- [ ] Bonus (nếu có) có định lượng trước/sau

# 6) LƯU KẾT QUẢ VÀO GITHUB (Colab)

Cell dưới đây **chỉ push data/outputs** (golden_dataset.csv, graphrag_eval_results.csv, graphrag_vs_flatrag_summary.csv).

File **notebook** (`.ipynb`) phải lưu riêng bằng **File > Save a copy in GitHub** trên Colab để giữ lại output các cell.

In [24]:
#@title FINAL — Commit & push data/outputs len GitHub (Colab)
if "REPO_DIR" in dir() and Path(REPO_DIR).exists():
    subprocess.run(["git", "-C", REPO_DIR, "add", "data", "outputs"], check=True)
    commit = subprocess.run(["git", "-C", REPO_DIR, "commit", "-m", "Add Day19 GraphRAG lab outputs"])
    if commit.returncode == 0:
        subprocess.run(["git", "-C", REPO_DIR, "push"], check=True)
        print("OK: da push data/outputs len GitHub.")
    else:
        print("Khong co gi moi de commit (co the da push roi).")
else:
    print("REPO_DIR khong ton tai -> tai file thu cong bang google.colab.files.download().")


REPO_DIR khong ton tai -> tai file thu cong bang google.colab.files.download().
